In [0]:
%sql
CREATE TABLE IF NOT EXISTS finance.liquidity.assumption_reference (
    framework STRING,
    assumption_category STRING,
    assumption_name STRING,
    assumption_value STRING,
    assumption_description STRING,
    effective_date DATE,
    expiration_date DATE,
    active_flag STRING,
    source_system STRING,
    owner STRING,
    comments STRING,
    created_by STRING,
    created_ts TIMESTAMP,
    modified_by STRING,
    modified_ts TIMESTAMP
)
USING DELTA;


In [0]:
%sql
DESCRIBE finance.liquidity.assumption_reference;

In [0]:
%sql
-- Run once only.
-- assumption_code already exists in the table.
-- Re-running this cell will generate a FIELD_ALREADY_EXISTS error.

%sql
-- ALTER TABLE finance.liquidity.assumption_reference
-- ADD COLUMNS (
--     assumption_code STRING
-- );

In [0]:
%sql
DESCRIBE finance.liquidity.assumption_reference;


In [0]:
%sql
INSERT INTO finance.liquidity.assumption_reference
(
    framework,
    assumption_category,
    assumption_code,
    assumption_name,
    assumption_value,
    assumption_description,
    effective_date,
    expiration_date,
    active_flag,
    source_system,
    owner,
    comments,
    created_by,
    created_ts,
    modified_by,
    modified_ts
)
VALUES

(
    'LCR',
    'Behavioral',
    'LCR_RUNOFF',
    'Retail Runoff Rate',
    '5%',
    'LCR Retail Deposit Runoff Rate',
    CURRENT_DATE(),
    NULL,
    'Y',
    'Treasury',
    'Treasury',
    NULL,
    CURRENT_USER(),
    CURRENT_TIMESTAMP(),
    CURRENT_USER(),
    CURRENT_TIMESTAMP()
),

(
    'ODM',
    'Behavioral',
    'ODM_ATTRITION',
    'Attrition Rate',
    '12%',
    'Customer Attrition Rate',
    CURRENT_DATE(),
    NULL,
    'Y',
    'Treasury',
    'Treasury',
    NULL,
    CURRENT_USER(),
    CURRENT_TIMESTAMP(),
    CURRENT_USER(),
    CURRENT_TIMESTAMP()
);


In [0]:
%sql
SELECT *
FROM finance.liquidity.assumption_reference;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW lcr_runoff_factors AS
SELECT * FROM (
  VALUES
    ('FSE Non-Operational NMD', CAST(1.00 AS DECIMAL(5,4))),
    ('Uninsured Reciprocal Retail TDs LT30', CAST(0.10 AS DECIMAL(5,4))),
    ('FSE Reciprocal Wholesale TD GT30', CAST(1.00 AS DECIMAL(5,4))),
    ('NFSE Reciprocal Wholesale TD GT30', CAST(0.40 AS DECIMAL(5,4))),
    ('NFSE Insured Traditional Wholesale TD LT30', CAST(0.20 AS DECIMAL(5,4))),
    ('Uninsured Reciprocal Retail TDs GT30', CAST(0.10 AS DECIMAL(5,4))),
    ('NFSE Insured Operational Deposits', CAST(0.05 AS DECIMAL(5,4))),
    ('NFSE Uninsured Traditional Wholesale TD LT30', CAST(0.40 AS DECIMAL(5,4))),
    ('PSE Secured Deposit', CAST(0.20 AS DECIMAL(5,4))),
    ('Insured Reciprocal Retail TDs LT30', CAST(0.05 AS DECIMAL(5,4))),
    ('NFSE Insured Traditional Wholesale TD GT30', CAST(0.10 AS DECIMAL(5,4))),
    ('Insured Reciprocal Retail TDs GT30', CAST(0.05 AS DECIMAL(5,4))),
    ('Bankruptcy Secured Deposit', CAST(0.20 AS DECIMAL(5,4))),
    ('Repo Secured Deposit', CAST(0.20 AS DECIMAL(5,4))),
    ('Stable Retail NMDs', CAST(0.03 AS DECIMAL(5,4))),
    ('NFSE Reciprocal Wholesale NMD', CAST(.40 AS DECIMAL(5,4))),
    ('Stable Retail TDs GT30', CAST(0.03 AS DECIMAL(5,4))),
    ('Brokered NMDs GT30', CAST(0.25 AS DECIMAL(5,4))),
    ('PNB Trust Secured Deposit', CAST(0.20 AS DECIMAL(5,4))),
    ('FSE Insured Operational Deposits', CAST(0.05 AS DECIMAL(5,4))),
     ('FSE Uninsured Operational Deposits', CAST(0.25 AS DECIMAL(5,4))),
    ('Brokered NMDs LT30', CAST(0.40 AS DECIMAL(5,4))),
    ('Uninsured Reciprocal Retail NMDs', CAST(0.10 AS DECIMAL(5,4))),
    ('Other Retail TDs LT30', CAST(0.10 AS DECIMAL(5,4))),
    ('NFSE Uninsured Operational Deposits', CAST(0.25 AS DECIMAL(5,4))),
    ('FSE Reciprocal Wholesale TD LT30', CAST(1.00 AS DECIMAL(5,4))),
    ('ATI Secured Deposit', CAST(0.20 AS DECIMAL(5,4))),
    ('FSE Reciprocal Wholesale NMD', CAST(1.00 AS DECIMAL(5,4))),
    ('Opioid Secured Deposit', CAST(0.20 AS DECIMAL(5,4))),
    ('Other Retail NMDs', CAST(0.10 AS DECIMAL(5,4))),
    ('Brokered CDs GT30', CAST(0.25 AS DECIMAL(5,4))),
    ('Other Retail TDs GT30', CAST(0.10 AS DECIMAL(5,4))),
    ('Stable Retail TDs LT30', CAST(0.03 AS DECIMAL(5,4))),
    ('Insured Reciprocal Retail NMDs', CAST(0.05 AS DECIMAL(5,4))),
    ('NFSE Reciprocal Wholesale TD LT30', CAST(1.00 AS DECIMAL(5,4))),
    ('Brokered CDs LT30', CAST(0.40 AS DECIMAL(5,4))),
    ('NFSE Uninsured Traditional Wholesale TD GT30', CAST(0.25 AS DECIMAL(5,4))),
    ('NFSE Insured Excess Operational Deposits', CAST(0.25 AS DECIMAL(5,4))),
    ('FSE Excess Operational Deposits', CAST(1.00 AS DECIMAL(5,4))),
    ('Other FSE Deposits', CAST(1.00 AS DECIMAL(5,4))),
    ('NFSE Uninsured Excess Operational Deposits', CAST(0.40 AS DECIMAL(5,4)))
) AS t(rollup, runoff_factor);

SELECT * FROM lcr_runoff_factors;

In [0]:
%sql
UPDATE finance.liquidity.assumption_reference
SET assumption_value = '7%',
    modified_by = CURRENT_USER(),
    modified_ts = CURRENT_TIMESTAMP()
WHERE assumption_code = 'LCR_RUNOFF';

In [0]:
%sql
SELECT
    framework,
    assumption_code,
    COUNT(*) AS row_count
FROM finance.liquidity.assumption_reference
GROUP BY framework, assumption_code
HAVING COUNT(*) > 1
ORDER BY framework, assumption_code;

In [0]:
%sql
SELECT
    framework,
    assumption_code,
    assumption_name,
    assumption_value,
    active_flag,
    effective_date
FROM finance.liquidity.assumption_reference
WHERE active_flag = 'Y'
ORDER BY framework, assumption_code;

In [0]:
%sql
SELECT
    framework,
    COUNT(*) AS assumption_count
FROM finance.liquidity.assumption_reference
GROUP BY framework
ORDER BY framework;

In [0]:
%sql
SELECT
    framework,
    COUNT(DISTINCT assumption_code) AS unique_assumption_count
FROM finance.liquidity.assumption_reference
GROUP BY framework
ORDER BY framework;

In [0]:
%sql
WITH ranked_assumptions AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY assumption_code
               ORDER BY modified_ts DESC
           ) AS rn
    FROM finance.liquidity.assumption_reference
)
SELECT
    framework,
    assumption_code,
    assumption_name,
    assumption_value,
    effective_date
FROM ranked_assumptions
WHERE rn = 1
ORDER BY framework;